In [7]:
import pandas as pd
import re
import os

# ============================================
# 1. COMBINE ALL MONTHLY FILES
# ============================================

files = {
    "July": "POWERSTAR JUL.xlsx",
    "August": "POWERSTAR AUG.xlsx",
    "September": "POWERSTAR SEP.xlsx",
    "October": "POWERSTAR OCT.xlsx",
    "November": "POWERSTAR NOV.xlsx",
    "December": "POWERSTAR  DEC.xlsx",
    "January": "POWERSTAR JAN.xlsx",
    "February": "POWERSTAR  FEB.xlsx"
}

df_list = []

for month, file in files.items():
    if os.path.exists(file):
        temp = pd.read_excel(file, sheet_name="DATA")
        temp["Month"] = month
        df_list.append(temp)
        print(f"Loaded {file}")
    else:
        print(f"❌ Missing file: {file}")

df = pd.concat(df_list, ignore_index=True)
print("\n✅ All files combined!")

# ============================================
# STANDARDIZE SUPPLIER (FIRST WORD – ORIGINAL)
# ============================================

df['Supplier_Std'] = (
    df['Supplier']
    .str.strip()
    .str.lower()
    .str.split()
    .str[0]
)

# ============================================
# 2. STANDARDIZE DEPARTMENT (ORIGINAL – UNTOUCHED)
# ============================================

def standardize_department(dept):
    if pd.isna(dept):
        return None

    dept = str(dept).lower().strip()
    dept = re.sub(r'[^a-z0-9\s]', ' ', dept)
    dept = re.sub(r'\s+', ' ', dept)

    if 'bleach' in dept:
        return 'Bleach'
    elif any(x in dept for x in ['light', 'candle']):
        return 'Candles'
    elif any(x in dept for x in [
        'dishwash', 'dishwahing', 'soaps dishwashing',
        'dish washing', 'd/wash', 'dwl', 'd/l'
    ]):
        return 'Dishwashing'
    elif 'fabric soften' in dept:
        return 'Fabric Softeners'
    elif any(x in dept for x in ['h/w', 'hand wash', 'handwash', 'h/wash', 'hand/w']):
        return 'Handwash'
    elif any(x in dept for x in ['ball', 'balls']):
        return 'Toilet Balls'
    elif any(x in dept for x in ['block', 'blocks']):
        return 'Air Freshener Blocks'
    elif any(x in dept for x in ['scour', 'scouring', 'scoring', 'scouring powder', 'scour powder']):
        return 'Scouring Powders'
    elif any(x in dept for x in [
        'toilet cleaner', 'toiletcleaner', 'toilet gel',
        'toilet liquid', 'toilet liquids',
        'toilet cleaner liquids', 'toilet cleanerliquid', 'toilet clener liquids'
    ]):
        return 'Toilet Cleaners'
    elif 'shower gel' in dept:
        return 'Body Wash'
    elif 'mineral water' in dept:
        return 'Water'
    elif 'water' in dept:
        return 'Water Purifier'
    elif 'air' in dept:
        return 'Air Fresheners'
    else:
        return dept.title()

df['Category'] = df['Department'].apply(standardize_department)
print("✅ Department standardization done!")

# ============================================
# 3. CATEGORY FIXES BASED ON DESCRIPTION
# (YOUR FULL LOGIC – NOTHING REMOVED)
# ============================================

df['Description'] = df['Description'].astype(str).str.lower().str.strip()
df['Category'] = df['Category'].astype(str).str.strip()

def fix_category(row):
    desc = row['Description']
    cat = row['Category']

    window_glass_keywords = [
        'magn glass cleaner','safish wdw cleaner','jet window cleaner',
        'topex lemon w/cleaner','topex lavender w/cleaner',
        'topex window cleaner','safisha glass','glass cleaner',
        'glasscln','glass clnr','glass cln','velvexwindow',
        'tintewindow','g/cleaner','g.cleaner','velvex w/cleaner','v.w/cleaner'
    ]

    if any(x in desc for x in window_glass_keywords):
        if '300ml' in desc or '320ml' in desc:
            return 'Window Cleaner'
        return 'Glass Cleaner'

    if any(x in desc for x in [
        'magnee pine', 'dry cleaner', 'tile', 'm/puro',
        'magnee cleaner','magnee lavender', 'magnee multi','claymulti'
    ]):
        return 'All Purpose'

    if any(x in desc for x in [
        'multipurpose', 'multi', 'multip', 'multp', 'multi/p',
        'multi-purp', 'm/purpose', 'multi-kleen', 'multikleen',
        'multkleen', 'multi kleen','mult-purp'
    ]):
        return 'Multipurpose'

    if 'block' in desc:
        return 'Air Freshener Blocks'

    if any(x in desc for x in ['disin', 'disinfectant', 'desinfectant','desnifectant']):
        return 'Disinfectant'

    if 'bleach' in cat.lower() or 'bleach' in desc:
        if any(x in desc for x in ['colour', 'color', 'colors', 'colours']):
            return 'Bleach - Colours'
        return 'Bleach - Regular'

    if any(x in desc for x in ['handwash', 'hand wash', 'h/wash', 'h wash', 'handwsh']) \
       and not any(x in desc for x in ['dish wash', 'dish washing', 'dish-wash', 'dish/w']):
        return 'Handwash'

    if any(x in desc for x in ['body wash', 'b/wash', 'b wash', 'shower gel', 'bdwash']):
        return 'Body Wash'

    if any(x in desc for x in ['aqua guard', 'aquarguard', 'waterguard', 'water guard']):
        return "Water Purifier"

    if any(x in desc for x in [
        'dw', 'dwscouring', 'dishwashing', 'dish w',
        'd/wash', 'dishwash', 'dish wash', 'dish washing', 'dish/w'
    ]) and not any(x in desc for x in ['handwash', 'body wash', 'bleach']):
        return 'Dishwashing'

    if any(x in desc for x in [
        'scouring powder','safisha scouring','safisha s/powder','scourer','scouring','scour'
    ]):
        return 'Scouring Powders'

    if 'magnee' in desc and cat.lower() == 'toilet cleaners':
        return 'All Purpose'

    if 'ball' in desc:
        return 'Toilet Balls'

    if 'glue' in desc:
        return 'Office Glue'

    if 'ariel' in desc:
        return 'Detergent'

    return cat

df['Category'] = df.apply(fix_category, axis=1)
print("✅ Category cleanup COMPLETE (all rules preserved)")

# ============================================
# 4. SUPPLIER FIXES (NEW – YOUR REQUEST)
# ============================================

def fix_supplier(row):
    desc = row['Description']

    if 'omo bleach' in desc:
        return 'Unilever Kenya Limited'
    if 'safisha bleach' in desc:
        return 'Sundries Bargains (Nairobi) Ltd'
    if any(x in desc for x in ['msafi bleach','msafi']):
        return 'Bidco Africa Ltd'
    if 'topex bleach' in desc:
        return 'Supersleek Limited'
    if desc.startswith('jik') or ' jik ' in desc:
        return 'Towfiq Kenya Limited'

    if any(x in desc for x in ['sta soft', 'sta-soft', 'stasoft', 'starsoft', 'sta-sof']):
        return 'Colgate'
    if any(x in desc for x in ['so soft', 's.soft', 's soft','sosoft','s-soft']):
        return 'Haco'
    if 'downy' in desc:
        return 'Hesbah Kenya Ltd'
    if any (x in desc for x in ['cuddles','cuddle']):
        return 'Supersleek Limited'

    if any(x in desc for x in [
        'safisha toilet cleaner','safisha t/c','safisha t.cleaner',
        'safisha t.clnr','safisha toilet','saf toilet','saf tlt', 'safisha colour','safish a', 'safisha regular','safisha reg','safisha lemon', 'safisha bleach'
    ]):
        return 'Sundries Bargains (Nairobi) Ltd'

    return row['Supplier']

df['Supplier'] = df.apply(fix_supplier, axis=1)

# ============================================
# 5. CLEAN SUPPLIER + RECREATE Supplier_Std
# ============================================

df['Supplier'] = df['Supplier'].str.title()

df['Supplier_Std'] = (
    df['Supplier']
    .str.strip()
    .str.lower()
    .str.split()
    .str[0]
)

print("✅ Supplier fixed & Supplier_Std rebuilt")

# ============================================
# 6. SAVE FINAL FILE
# ============================================

df.to_excel("POWERSTAR_FINAL_CLEANED.xlsx", index=False)
print("\n🎉 FINAL FILE SAVED – NOTHING LEFT BEHIND")


Loaded POWERSTAR JUL.xlsx
Loaded POWERSTAR AUG.xlsx
Loaded POWERSTAR SEP.xlsx
Loaded POWERSTAR OCT.xlsx
Loaded POWERSTAR NOV.xlsx
Loaded POWERSTAR  DEC.xlsx
Loaded POWERSTAR JAN.xlsx
Loaded POWERSTAR  FEB.xlsx

✅ All files combined!
✅ Department standardization done!
✅ Category cleanup COMPLETE (all rules preserved)
✅ Supplier fixed & Supplier_Std rebuilt

🎉 FINAL FILE SAVED – NOTHING LEFT BEHIND


In [ ]:
import pandas as pd

# Load your combined file
df = pd.read_excel("POWERSTAR_FINAL_CLEANED.xlsx")

# Create a mapping dictionary for handlers
handler_map = {
    "Express": "Isabella Wambui",
    "Hyper": "N/A",
    "Jambo": "Stephen Otieno",
    "Kangari": "Margaret Njeri",
    "Kasarani": "Linet Atieno",
    "Kikuyu": "Dorcas Paul",
    "Kinoo": "Dorcas Paul",
    "Kitengela": "Peris Wavinya",
    "Mini": "Stephen Otieno",
    "Vasha": "Francis Murage",
    "Zimmerman": "Isabella Wambui"
}

# Create a new column 'Handler'
df["Handler"] = df["Branch"].map(handler_map)

# Save updated file
df.to_excel("POWERSTAR_HANDLERS_ADDED.xlsx", index=False)


In [9]:
import pandas as pd

# Load the file from the previous step
df = pd.read_excel("POWERSTAR_HANDLERS_ADDED.xlsx")

# Create region mapping (example — update to your real regions)
region_map = {
    "Express": "Nairobi",
    "Hyper": "Nairobi",
    "Jambo": "Murang’a",
    "Kangari": "Murang’a",
    "Kasarani": "Nairobi",
    "Kikuyu": "Kiambu",
    "Kinoo": "Kiambu",
    "Kitengela": "Kajiado",
    "Mini": "Murang’a",
    "Vasha": "Nakuru",
    "Zimmerman": "Nairobi"
}

# Create a new column 'Region'
df["Region"] = df["Branch"].map(region_map)

# Save final file
df.to_excel("POWERSTAR_COMBINED_DATA.xlsx", index=False) 
